In [1]:
###Next Steps:
#Adding rewrite rules with conditions
from term import Term, Function_Symbol, Constant, Variable, get_constant, get_variable, build_term, function_symbol_registry, constant_registry, variable_registry, constant_order, generate_constant_order 
from weights import get_weight, vector_comparison_greater_than
from rewrite import normal_form
from rules import load_rules_from_file
import re
import parsley

t1 = Term.parse("X ^2 + 2*X ^ 2 + 2 +3" )



###This code is duplicated from term.py:
sorted_constant_keys = sorted(constant_registry.keys())
constant_order = {}
for index, key in enumerate(sorted_constant_keys, start=1):
    constant_order[key] = index
### End duplication
ruleset = load_rules_from_file("rewrite_rules.txt")

print(f"{constant_order=} \n")

t1_vector_metric = get_weight(t1, function_symbol_registry, constant_order, static_analysis=False)
print(f"{t1_vector_metric=}")
normal_form_t1 = normal_form(ruleset, t1, function_symbol_registry, constant_order)
print(f"Normal form of {t1} is {normal_form_t1}.")
t1_normal_form_vector_metric = get_weight(normal_form_t1, function_symbol_registry, constant_order, static_analysis=False)
print(f"{t1_normal_form_vector_metric=} \n")


constant_order={'X': 1} 

t1_vector_metric=[[0, -1], [2, 2, -1], [1, 1, -1], [0, -1], [0], [0, 0, 0, 0, -1], [0, -1], [11, -1], [2, 1, 0, -1], [5]]
Normal form of +(+(+(^(X, quote(2)), *(quote(2), ^(X, quote(2)))), quote(2)), quote(3)) is +(quote(5), *(quote(3), *(X, X))).
t1_normal_form_vector_metric=[[0, -1], [-1], [-1], [0, -1], [0], [0, -1], [0, 0, -1], [0, -1], [0, -1], [2]] 



In [ ]:
# ==========================================
# PARALLEL CONFLUENCE ANALYSIS (DEBUG ENABLED)
# ==========================================

import concurrent.futures
from rules import load_rules_from_file
from confluence import find_critical_pairs, ConfluenceCache, check_group_confluence_task
from term import function_symbol_registry, constant_order
from collections import defaultdict
from tqdm.notebook import tqdm

# Configuration
NUM_CORES = 20 

print("\nLoading rules...")
ruleset = load_rules_from_file("rewrite_rules.txt")
cache = ConfluenceCache()

if not ruleset:
    print("No rules loaded.")
else:
    print(f"Ruleset loaded: {len(ruleset)} rules.")
    
    # 1. Find Critical Pairs
    print("Finding Critical Pairs...")
    all_cps = find_critical_pairs(ruleset)
    print(f"Found {len(all_cps)} total critical pairs.")
    
    # 2. Group CPs
    cp_groups = defaultdict(list)
    for cp in all_cps:
        cp_groups[(cp.rule1, cp.rule2)].append(cp)
        
    # 3. Prepare Tasks
    tasks = []
    skipped_count = 0
    
    print("Preparing tasks...")
    for (r1, r2), cps_in_group in cp_groups.items():
        if cache.is_pair_checked(r1, r2):
            skipped_count += 1
            continue
            
        task_args = (r1, r2, cps_in_group, ruleset, function_symbol_registry, constant_order)
        tasks.append(task_args)
        
    print(f"Skipped {skipped_count} previously verified groups.")
    print(f"Queueing {len(tasks)} groups for analysis on {NUM_CORES} cores...")
    
    # 4. Run in Parallel
    system_confluent = True
    
    with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
        future_to_group = {executor.submit(check_group_confluence_task, task): task for task in tasks}
        
        for future in tqdm(concurrent.futures.as_completed(future_to_group), total=len(tasks), desc="Analyzing Groups"):
            try:
                # Unpack 4 values now: r1, r2, success, message
                r1, r2, success, message = future.result()
                
                if success:
                    cache.mark_pair_checked(r1, r2)
                else:
                    print(f"\n[FAIL] Interaction not joinable: {r1.LHS} vs {r2.LHS}")
                    # Print the detailed debug info returned by the worker
                    if message:
                        print(message)
                    system_confluent = False
                    
            except Exception as e:
                print(f"\n[ERROR] A worker process crashed: {e}")
                import traceback
                traceback.print_exc()
                system_confluent = False

    cache.save()

    print("\n" + "="*30)
    if system_confluent:
        print("SYSTEM IS LOCALLY CONFLUENT")
    else:
        print("SYSTEM IS NOT CONFLUENT")
    print("="*30)